Feature Engineering

In [23]:
#Importing the libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import precision_recall_curve


In [2]:
#Loading the dataset
data = pd.read_csv('s:/Projects/Customer Churn ML Model and API/data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [3]:
#Dropping the customerID column as it is not needed for analysis
data.drop('customerID', axis=1, inplace=True)

#Converting the TotalCharges column to numeric, coercing errors to NaN
data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce')

#Dropping rows with NaN values in the TotalCharges column
data.dropna(subset=['TotalCharges'], inplace=True)

In [4]:
#Converting the target variable 'Churn' to binary format
data['Churn'] = data['Churn'].map({'Yes': 1, 'No': 0})

In [5]:
print(data.head())
print(data.shape)

   gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  Female              0     Yes         No       1           No   
1    Male              0      No         No      34          Yes   
2    Male              0      No         No       2          Yes   
3    Male              0      No         No      45           No   
4  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity OnlineBackup  \
0  No phone service             DSL             No          Yes   
1                No             DSL            Yes           No   
2                No             DSL            Yes          Yes   
3  No phone service             DSL            Yes           No   
4                No     Fiber optic             No           No   

  DeviceProtection TechSupport StreamingTV StreamingMovies        Contract  \
0               No          No          No              No  Month-to-month   
1              Yes          No  

In [6]:
print(data.info())

<class 'pandas.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7032 non-null   str    
 1   SeniorCitizen     7032 non-null   int64  
 2   Partner           7032 non-null   str    
 3   Dependents        7032 non-null   str    
 4   tenure            7032 non-null   int64  
 5   PhoneService      7032 non-null   str    
 6   MultipleLines     7032 non-null   str    
 7   InternetService   7032 non-null   str    
 8   OnlineSecurity    7032 non-null   str    
 9   OnlineBackup      7032 non-null   str    
 10  DeviceProtection  7032 non-null   str    
 11  TechSupport       7032 non-null   str    
 12  StreamingTV       7032 non-null   str    
 13  StreamingMovies   7032 non-null   str    
 14  Contract          7032 non-null   str    
 15  PaperlessBilling  7032 non-null   str    
 16  PaymentMethod     7032 non-null   str    
 17  MonthlyChar

In [7]:
#Extracting str columns for encoding
str_cols = data.select_dtypes(include='str').columns
print(str_cols)

Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='str')


In [8]:
#Converting the categorical variables to dummy variables
data = pd.get_dummies(data, columns=str_cols, drop_first=True)

In [9]:
print(data.head())
print(data.shape)

   SeniorCitizen  tenure  MonthlyCharges  TotalCharges  Churn  gender_Male  \
0              0       1           29.85         29.85      0        False   
1              0      34           56.95       1889.50      0         True   
2              0       2           53.85        108.15      1         True   
3              0      45           42.30       1840.75      0         True   
4              0       2           70.70        151.65      1        False   

   Partner_Yes  Dependents_Yes  PhoneService_Yes  \
0         True           False             False   
1        False           False              True   
2        False           False              True   
3        False           False             False   
4        False           False              True   

   MultipleLines_No phone service  ...  StreamingTV_No internet service  \
0                            True  ...                            False   
1                           False  ...                            Fa

In [10]:
#Spliting the data into training and testing sets

x = data.drop('Churn', axis=1)
y = data['Churn']

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    stratify=y,
    random_state=9
)


In [11]:
#Printing numetic columns for scaling
num_cols = data.select_dtypes(include='number').columns
print(num_cols)

num_cols = num_cols.drop('Churn')
print(num_cols)

num_cols = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']


Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn'], dtype='str')
Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='str')


In [12]:
#Performing feature scaling using StandardScaler

scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train[num_cols])
x_test_scaled = scaler.transform(x_test[num_cols])

x_train = x_train.copy()
x_test = x_test.copy()

x_train[num_cols] = x_train_scaled
x_test[num_cols] = x_test_scaled

print(x_train.head())
print(x_test.head())

      SeniorCitizen    tenure  MonthlyCharges  TotalCharges  gender_Male  \
4998       2.232497  1.195531        1.521077      2.087109         True   
6460      -0.447929  0.462252        1.182589      0.920478         True   
2600      -0.447929 -1.289471        0.172103     -0.981164        False   
1258      -0.447929  0.584465       -1.485493     -0.552289        False   
5922      -0.447929  1.602909        1.338559      2.331578         True   

      Partner_Yes  Dependents_Yes  PhoneService_Yes  \
4998        False           False              True   
6460        False           False              True   
2600        False           False              True   
1258        False           False              True   
5922         True            True              True   

      MultipleLines_No phone service  MultipleLines_Yes  ...  \
4998                           False               True  ...   
6460                           False              False  ...   
2600                

In [13]:
#Printing mean and standard deviation of the numeric columns
print(x_train[num_cols].mean())
print(x_train[num_cols].std())

SeniorCitizen     7.326485e-17
tenure           -1.326346e-17
MonthlyCharges   -1.313715e-16
TotalCharges     -4.168517e-17
dtype: float64
SeniorCitizen     1.000089
tenure            1.000089
MonthlyCharges    1.000089
TotalCharges      1.000089
dtype: float64


In [14]:
#Training and comparing models

#Logistic Regression
model_lr = LogisticRegression(class_weight='balanced')

model_lr.fit(x_train, y_train)

#y_pred_lr = model_lr.predict(x_test)

y_prob = model_lr.predict_proba(x_test)[:, 1]
#print(y_prob)

precision, recall, thresholds = precision_recall_curve(y_test, y_prob)

f1_scores = 2 * (precision[1:] * recall[1:]) / (precision[1:] + recall[1:])

best_idx = np.argmax(f1_scores)

best_threshold = thresholds[best_idx]

print("Best Threshold:", best_threshold)
print("Best F1 Score:", f1_scores[best_idx])

y_pred_lr = (y_prob >= best_threshold).astype(int)

print("Logistic Regression Classification Report:")
print(accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

#As our model is good at finding which customer stayed but not good at 
# finding which customer left, we will try class balancing techniques
#  to improve the recall score of the model.

#After trying class balancing techniques, we can see that the recall 
# score of the model has improved but the precision score has decreased. 
# This is a common trade-off when using class balancing techniques. 
# The model is now better at finding which customers left but it is 
# also more likely to incorrectly classify some customers as leaving
#  when they actually stayed. 
# So, we find the best threshhold for our model using F1 score as our metric.

#So, after finding the best threshold for our model, we can see that the 
# recall score has improved significantly but the precision score has 
# decreased. This is a common trade-off when using class balancing 
# techniques. The model is now better at finding which customers left 
# but it is also more likely to incorrectly classify some customers as 
# leaving when they actually stayed.


"""
Results beofre class balancing:
Logistic Regression Classification Report:
0.7981520966595593
              precision    recall  f1-score   support

           0       0.84      0.89      0.87      1033
           1       0.65      0.53      0.58       374

    accuracy                           0.80      1407
   macro avg       0.74      0.71      0.73      1407
weighted avg       0.79      0.80      0.79      1407

Results after class balancing:
Logistic Regression Classification Report:
0.6275764036958067
              precision    recall  f1-score   support

           0       0.96      0.52      0.67      1033
           1       0.41      0.93      0.57       374

    accuracy                           0.63      1407
   macro avg       0.68      0.73      0.62      1407
weighted avg       0.81      0.63      0.64      1407
"""

Best Threshold: 0.5133372412085893
Best F1 Score: 0.6353944562899786
Logistic Regression Classification Report:
0.7562189054726368
              precision    recall  f1-score   support

           0       0.91      0.74      0.82      1033
           1       0.53      0.80      0.63       374

    accuracy                           0.76      1407
   macro avg       0.72      0.77      0.73      1407
weighted avg       0.81      0.76      0.77      1407



'\nResults beofre class balancing:\nLogistic Regression Classification Report:\n0.7981520966595593\n              precision    recall  f1-score   support\n\n           0       0.84      0.89      0.87      1033\n           1       0.65      0.53      0.58       374\n\n    accuracy                           0.80      1407\n   macro avg       0.74      0.71      0.73      1407\nweighted avg       0.79      0.80      0.79      1407\n\nResults after class balancing:\nLogistic Regression Classification Report:\n0.6275764036958067\n              precision    recall  f1-score   support\n\n           0       0.96      0.52      0.67      1033\n           1       0.41      0.93      0.57       374\n\n    accuracy                           0.63      1407\n   macro avg       0.68      0.73      0.62      1407\nweighted avg       0.81      0.63      0.64      1407\n'

In [22]:
#Random Forest Classifier

model_rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=9,
)

model_rf.fit(x_train, y_train)

y_prob_rf = model_rf.predict_proba(x_test)[:, 1]


precision, recall, thresholds = precision_recall_curve(
    y_test, 
    y_prob_rf)

f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1])

best_idx = np.argmax(f1_scores)

best_threshold = thresholds[best_idx]

print("Best Threshold:", best_threshold)

y_pred_rf = (y_prob_rf >= best_threshold).astype(int)

print("Random Forest Classification Report:")
print(accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

#Checking feature importance of the model
importances = pd.DataFrame({
    'feature': x_train.columns,
    'importance': model_rf.feature_importances_
    })

importances.sort_values(by='importance', ascending=False, inplace=True)

print(importances)

#Randdom forest classifier shower worse results than 
# logistic regression classifier. So, we will try to find the best 
# threshold for our model using F1 score as our metric. But before that,
# we will try to evualte again by defineing max depth, min samples split 
# and min samples leaf parameters for our model.

#After trying different hyperparameter tuning techniques, we can see that the
# recall score of the model has improved but the precision score has decreased.

#After trying to find the best threshold for our model using F1 score as 
# our metric, we can see that the recall score has improved significantly 
# but the precision score has decreased. This is a common trade-off when 
# using class balancing techniques. The model is now better at finding 
# which customers left but it is also more likely to incorrectly 
# classify some customers as leaving when they actually stayed.



"""
Results before class balancing:
Random Forest Classification Report:
0.7874911158493249
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1033
           1       0.63      0.49      0.55       374

    accuracy                           0.79      1407
   macro avg       0.73      0.69      0.71      1407
weighted avg       0.78      0.79      0.78      1407

Results after class balancing:
Random Forest Classification Report:
0.7924662402274343
              precision    recall  f1-score   support

           0       0.83      0.90      0.86      1033
           1       0.64      0.49      0.56       374

    accuracy                           0.79      1407
   macro avg       0.74      0.70      0.71      1407
weighted avg       0.78      0.79      0.78      1407

Results after class balancing and hyperparameter tuning:
Random Forest Classification Report:
0.7633262260127932
              precision    recall  f1-score   support

           0       0.90      0.77      0.83      1033
           1       0.54      0.75      0.63       374

    accuracy                           0.76      1407
   macro avg       0.72      0.76      0.73      1407
weighted avg       0.80      0.76      0.77      1407

"""

Best Threshold: 0.4619381591956552
Random Forest Classification Report:
0.7555081734186212
              precision    recall  f1-score   support

           0       0.91      0.74      0.82      1033
           1       0.53      0.80      0.64       374

    accuracy                           0.76      1407
   macro avg       0.72      0.77      0.73      1407
weighted avg       0.81      0.76      0.77      1407

                                  feature  importance
1                                  tenure    0.189939
3                            TotalCharges    0.133883
25                      Contract_Two year    0.114678
2                          MonthlyCharges    0.091506
10            InternetService_Fiber optic    0.073068
28         PaymentMethod_Electronic check    0.047747
24                      Contract_One year    0.046401
13                     OnlineSecurity_Yes    0.029946
19                        TechSupport_Yes    0.024414
26                   PaperlessBilling_Yes 

'\nResults before class balancing:\nRandom Forest Classification Report:\n0.7874911158493249\n              precision    recall  f1-score   support\n\n           0       0.83      0.89      0.86      1033\n           1       0.63      0.49      0.55       374\n\n    accuracy                           0.79      1407\n   macro avg       0.73      0.69      0.71      1407\nweighted avg       0.78      0.79      0.78      1407\n\nResults after class balancing:\nRandom Forest Classification Report:\n0.7924662402274343\n              precision    recall  f1-score   support\n\n           0       0.83      0.90      0.86      1033\n           1       0.64      0.49      0.56       374\n\n    accuracy                           0.79      1407\n   macro avg       0.74      0.70      0.71      1407\nweighted avg       0.78      0.79      0.78      1407\n\nResults after class balancing and hyperparameter tuning:\nRandom Forest Classification Report:\n0.7633262260127932\n              precision    r

In [28]:
#Gradient Boosting Classifier

model_gb = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    random_state=9
)

model_gb.fit(x_train, y_train)

y_prob_gb = model_gb.predict_proba(x_test)[:, 1]

precision, recall, thresholds = precision_recall_curve(
    y_test,
    y_prob_gb)

f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1])

best_idx = np.argmax(f1_scores)

best_threshold = thresholds[best_idx]

y_preb_gb = (y_prob_gb >= best_threshold).astype(int)

print("Gradient Boosting Classification Report:")
print(accuracy_score(y_test, y_preb_gb))
print(classification_report(y_test, y_preb_gb))


#Gradient boostimng classifer perfomrs as we expected. 
# So, we will try to find the best threshold for our model using F1 score 
# as our metric. But before that, we will try to evualte again by 
# defineing max depth, min samples split and min samples leaf parameters 
# for our model.

#After hyperparameter tuning and threshold optimization, Gradient Boosting
#  achieved comparable performance to Random Forest, with an F1-score of
#  approximately 0.64 for the churn class and a recall of 0.81. 
# This indicates that boosting models are effective for capturing 
# churn behavior when probability thresholds are optimized.

"""
#Results before class balancing:
Gradient Boosting Classification Report:
0.7960199004975125
              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1033
           1       0.64      0.52      0.58       374

    accuracy                           0.80      1407
   macro avg       0.74      0.71      0.72      1407
weighted avg       0.79      0.80      0.79      1407

#Results after F1 score threshold tuning:
Gradient Boosting Classification Report:
0.7569296375266524
              precision    recall  f1-score   support

           0       0.91      0.74      0.82      1033
           1       0.53      0.81      0.64       374

    accuracy                           0.76      1407
   macro avg       0.72      0.77      0.73      1407
weighted avg       0.81      0.76      0.77      1407
"""

Gradient Boosting Classification Report:
0.7569296375266524
              precision    recall  f1-score   support

           0       0.91      0.74      0.82      1033
           1       0.53      0.81      0.64       374

    accuracy                           0.76      1407
   macro avg       0.72      0.77      0.73      1407
weighted avg       0.81      0.76      0.77      1407



'\n#Results before class balancing:\nGradient Boosting Classification Report:\n0.7960199004975125\n              precision    recall  f1-score   support\n\n           0       0.84      0.90      0.87      1033\n           1       0.64      0.52      0.58       374\n\n    accuracy                           0.80      1407\n   macro avg       0.74      0.71      0.72      1407\nweighted avg       0.79      0.80      0.79      1407\n\n#Results after F1 score threshold tuning:\nGradient Boosting Classification Report:\n0.7569296375266524\n              precision    recall  f1-score   support\n\n           0       0.91      0.74      0.82      1033\n           1       0.53      0.81      0.64       374\n\n    accuracy                           0.76      1407\n   macro avg       0.72      0.77      0.73      1407\nweighted avg       0.81      0.76      0.77      1407\n'